# Mortality classification tasks - MACSS

In [ ]:
import yaml
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import joblib
import os

# Load all models
ALL_MODELS = yaml.safe_load(open("../config.yml"))["models"]

# Filter for MACSS Mortality models
relevant_keys = [k for k in ALL_MODELS.keys() if k.startswith("macss_") and "los" not in k and "survival" not in k]
print(f"Models to train: {relevant_keys}")

## Index-Specific Train/Test Splits

Create views of the train/test splits which have the comorbidities for each index present.

### Data Aggregation

The training data can be quite large which results in extremely large inputs for the models. For example, the MACSS has 100 comorbidities so the training data is a N x 100 matrix where N is the number of rows.

To improve the training time for our models, we can 'compress' the data and represent it by identifying each unique combination of comorbidities and the number of times it occurred.

For example, given the following row-level data:

| COMORB_1 | COMORB_2 | COMORB_3 |
| -------- | -------- | -------- |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    0     |
|    1     |    0     |    1     |
|    1     |    0     |    1     |

We would represent it as:

| COMORB_1 | COMORB_2 | COMORB_3 | N |
| -------- | -------- | -------- | - |
|    1     |    1     |    1     | 7 |
|    1     |    1     |    0     | 1 |
|    1     |    0     |    1     | 2 |

In [ ]:
for model_key in relevant_keys:
    MODEL_CONFIG = ALL_MODELS[model_key]
    print(f"\nProcessing {model_key} (Target: {MODEL_CONFIG['class']})...")
    
    # Load Data
    macss_training = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_training']}")
    macss_testing = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_testing']}")
    
    # Data Aggregation
    macss_training_agg = macss_training.groupby(list(macss_training.columns),dropna=False).size().reset_index(name='N')
    macss_testing_agg = macss_testing.groupby(list(macss_testing.columns),dropna=False).size().reset_index(name='N')
    
    # Define input columns
    input_cols = [i for i in macss_training_agg.columns if "C_" in i]
    
    # Train
    clf = LogisticRegression(penalty=None)
    clf.fit(macss_training_agg[input_cols], macss_training_agg[MODEL_CONFIG["class"]], sample_weight=macss_training_agg['N'])
    
    # Predict
    y_train_pred_proba = clf.predict_proba(macss_training_agg[input_cols])
    y_train_true = macss_training_agg[MODEL_CONFIG["class"]]
    
    y_test_pred_proba = clf.predict_proba(macss_testing_agg[input_cols])
    y_test_true = macss_testing_agg[MODEL_CONFIG["class"]]
    
    # Metrics
    fpr_train, tpr_train, _ = roc_curve(y_train_true, y_train_pred_proba[:,1], sample_weight=macss_training_agg['N'])
    print(f"  Training AUC: {auc(fpr_train, tpr_train):.4f}")
    
    fpr_test, tpr_test, _ = roc_curve(y_test_true, y_test_pred_proba[:,1], sample_weight=macss_testing_agg['N'])
    print(f"  Test AUC: {auc(fpr_test, tpr_test):.4f}")
    
    # Save
    if not os.path.exists('../models'):
        os.makedirs('../models')
    
    save_path = f'../models/mort_macss_{MODEL_CONFIG["class"]}.joblib'
    joblib.dump(clf, save_path)
    print(f"  Model saved to {save_path}")